In [1]:
from platform import python_version
print(python_version())

3.11.14


### Tahoe

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

from scipy.stats import spearmanr


import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_prism

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism')

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/deconv.h5ad (8.1 MB)
1918


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [12]:
res.cell_type_expression("Ductal cell type 1").shape

(1918, 153)

In [13]:
res.cell_type_expression("Ductal cell type 2").shape

(1918, 153)

### 2. theta is now fixed -> expand Z to every gene

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file


In [15]:
print(dfn_tumor.shape)
dfn_tumor.head(3)

(60616, 134)


,geneid,symbol,biotype,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,...,T-TCGA-HZ-7289,T-TCGA-3A-A9IN,T-TCGA-3A-A9IS,T-TCGA-2L-AAQM,T-TCGA-3A-A9IR,T-TCGA-3A-A9IV,T-TCGA-3A-A9IO,T-TCGA-2J-AABT,T-TCGA-H6-A45N,T-TCGA-3A-A9IJ
0,ENSG00000000003,TSPAN6,protein_coding,1486,2083,1558,546,1208,648,896,...,2506,442,38,299,77,158,394,659,1294,395
1,ENSG00000000005,TNMD,protein_coding,12,97,15,1,14,2,5,...,2,172,4,2,1,2,14,3,3,0
2,ENSG00000000419,DPM1,protein_coding,1330,1521,1499,986,1388,974,649,...,1638,940,1372,1008,1493,1059,922,839,717,1034


### Bulk - by geneid

In [16]:
df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        gene_key="geneid", force=False, verbose=verbose)

#--- reference geneid --> 
gene_map = prism.load_gene_map("geneid")
print(df_bulk.shape)
df_bulk.head(2)

(27177, 153)


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,1486,2083,1558,546,1208,648,896,1532,821,1217,...,891,1063,1261,1821,554,1244,977,1576,3738,369
ENSG00000000005,12,97,15,1,14,2,5,6,3,11,...,0,7,2,6,1,1,3,39,4,5


In [17]:
# cbio.plot_boxplot_expression(df_bulk, do_log10=True, fname_corr = "Expression across normal samples")

### Prism single-cell data reference - Peng 2019

In [18]:
force=False
verbose=True

fname = "count-matrix.txt"
adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

fname_ad = fname.replace('.txt', '.h5ad')
filename_ad = prism.root_prism / fname_ad
compression = "gzip"


verbose=True
fname_celltype = "all_celltype.txt"
adata_ct = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

adata_ct

57,530 cells x 24,005 genes | obs: []
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 2447
Acinar cell            1935
Endocrine cell          729
Name: count, dtype: int64


AnnData object with n_obs × n_vars = 57530 × 24005
    obs: 'cluster', 'cell_type', 'cell_state'

### Reference

In [19]:
ref, s2t = prism.pseudobulk_reference(adata_ct)

print(ref.shape)
ref.head(3)

(10, 23789)


gene,AL627309.1,AP006222.2,RP11-206L10.3,RP11-206L10.2,RP11-206L10.9,LINC00115,FAM41C,RP11-54O7.3,SAMD11,NOC2L,...,RP11-133N21.10,TMEM5-AS1,RP11-175P13.2,TBC1D4-AS1,RP11-329J18.3,RP11-227G15.2,RP11-686D22.5,KRT222,RP11-799B12.2,CTC-429C10.2
cell_state,,,,,,,,,,,,,,,,,,,,,
Fibroblast cell,64.0,1612.0,42.0,22.0,30.0,214.0,102.0,103.0,1246.0,2337.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Stellate cell,44.0,903.0,25.0,16.0,30.0,162.0,55.0,13.0,112.0,1719.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Macrophage cell,141.0,696.0,81.0,29.0,22.0,182.0,124.0,5.0,35.0,1293.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Bulk symbols

In [20]:
bulk_symbs = pd.read_csv(prism.root_prism / "bulk_matrix.tsv", sep="\t", index_col=0, usecols=[0])

### Get Harmonize

In [21]:
verbose=True

ref_new, df_to_from = prism.get_harmonize_reference_to_ensembl(ref=ref, verbose=verbose)

print(ref_new.shape)
print(df_to_from.status.value_counts())
ref_new.head(2)

Table opened ((10, 17540)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/ref_new_geneid.tsv'
Table opened ((23789, 5)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/to_from_table.tsv'
(10, 17540)
status
ok               17540
no_ensembl_id     6249
Name: count, dtype: int64


,ENSG00000225880,ENSG00000187634,ENSG00000188976,ENSG00000187961,ENSG00000187583,ENSG00000188290,ENSG00000187608,ENSG00000188157,ENSG00000237330,ENSG00000131591,...,ENSG00000167355,ENSG00000184999,ENSG00000235910,ENSG00000157335,ENSG00000257008,ENSG00000166573,ENSG00000268182,ENSG00000254453,ENSG00000226245,ENSG00000213424
cell_state,,,,,,,,,,,,,,,,,,,,,
Fibroblast cell,214.0,1246.0,2337.0,115.0,90.0,4981.0,18665.0,1758.0,11.0,274.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Stellate cell,162.0,112.0,1719.0,52.0,26.0,9730.0,16767.0,1611.0,2.0,172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [22]:
df_to_from2 = df_to_from[~pd.isnull(df_to_from.geneid)]
df_to_from2.head(3)

,ref_symbol,current_symbol,geneid,status,renamed
5,LINC00115,LINC00115,ENSG00000225880,ok,False
8,SAMD11,SAMD11,ENSG00000187634,ok,False
9,NOC2L,NOC2L,ENSG00000188976,ok,False


### gene_subset = select_genes(ref_new)

Informative-gene selection (analogue of BayesPrism's get.exp.stat + select.marker).  
Keeps genes with a strong one-vs-rest log2 fold change in the CPM-normalised pseudobulk reference.

Restricting to ~2-5k informative genes is what makes the fixed point both fast and stable;   
it barely changes theta but massively reduces the influence of protocol-driven genes.

In [23]:
gene_subset = prism.select_genes(ref_new)
len(gene_subset)

1918

### get bayesprism

In [24]:
verbose=True
force=False

meta_desc = dict(reference="Peng2019_CRA001160",
              cohorts=["TCGA-PAAD", "CPTAC3"],
              strand="unstranded",
              method="InstaPrism")

res = prism.run_bayesprism(df_bulk=df_bulk, ref=ref_new, state_to_type=s2t,
                           gene_subset=gene_subset, meta_desc=meta_desc, force=False, verbose=verbose)

type(res)

Loaded /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/prism/deconv.h5ad (8.1 MB)


libs.prism_lib.DeconvResult

### Zfull ~ has 17540 genes like ref_new

- bulk has 27177 genes
- selecd ref_new 1918
- ref_new has 17540 genes

In [25]:
res.Z.shape, df_bulk.shape, ref_new.shape

((153, 10, 1918), (27177, 153), (10, 17540))

In [26]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref_new)
print(Zfull.shape)

(153, 10, 17540)


### Counts per cell

In [27]:
dic = {}

for cell_state in res.states:
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z
    print(cell_state, Z.shape)

print("")
Z.head(3)


Fibroblast cell (17540, 153)
Stellate cell (17540, 153)
Macrophage cell (17540, 153)
Endothelial cell (17540, 153)
T cell (17540, 153)
B cell (17540, 153)
Ductal cell type 2 (17540, 153)
Endocrine cell (17540, 153)
Ductal cell type 1 (17540, 153)
Acinar cell (17540, 153)



,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
ENSG00000000003,55.465,NaN,NaN,41.988,60.980,NaN,NaN,NaN,NaN,NaN,...,NaN,25.722,34.816,NaN,NaN,NaN,NaN,NaN,79.945,NaN
ENSG00000000005,0.207,NaN,NaN,0.029,0.237,NaN,NaN,NaN,NaN,NaN,...,NaN,0.147,0.032,NaN,NaN,NaN,NaN,NaN,0.040,NaN
ENSG00000000419,25.238,NaN,NaN,28.842,26.863,NaN,NaN,NaN,NaN,NaN,...,NaN,14.213,27.300,NaN,NaN,NaN,NaN,NaN,17.400,NaN


In [28]:
i=0
key = list(dic.keys())[i]

print(key)
Z = dic[key]
Z.head(3)

Fibroblast cell


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
ENSG00000000003,32.066,58.012,43.613,32.366,34.058,31.881,35.654,54.328,30.481,40.203,...,NaN,16.647,16.364,NaN,NaN,NaN,27.003,NaN,53.037,NaN
ENSG00000000005,0.156,1.310,0.271,0.029,0.172,0.058,0.136,0.085,0.060,0.182,...,NaN,0.124,0.020,NaN,NaN,NaN,0.119,NaN,0.035,NaN
ENSG00000000419,41.234,43.095,42.380,62.828,42.398,43.955,25.719,43.317,42.273,46.569,...,NaN,25.995,36.263,NaN,NaN,NaN,21.046,NaN,32.622,NaN


### tumour-vs-normal design

The tumour-vs-normal design is also the standard one for deconvolution benchmarking:

your normals give an internal check that θ_malignant ≈ 0 where it should be,  
which is the only ground truth you have.

Worth confirming after the ENSG rebuild:

In [29]:
n_idx = [s for s in res.theta.index if s.startswith("N-")]
res.theta.loc[n_idx, "Ductal cell type 2"].describe()

count    1.500e+01
mean     2.915e-01
std      4.165e-01
min      5.772e-36
25%      2.223e-03
50%      6.589e-02
75%      5.626e-01
max      9.846e-01
Name: Ductal cell type 2, dtype: float64

In [30]:
n_idx = [s for s in res.theta.index if s.startswith("T-")]
res.theta.loc[n_idx, "Ductal cell type 2"].describe()

count    1.300e+02
mean     3.452e-01
std      2.559e-01
min      2.706e-22
25%      1.450e-01
50%      2.982e-01
75%      4.965e-01
max      1.000e+00
Name: Ductal cell type 2, dtype: float64

### AUC 

AUC is symmetric about 0.5 under label flipping. Your loop scored tumours as the positive class, so acinar's 0.318 means "acinar fraction predicts tumour" *poorly* — but the same number says acinar predicts **normal** at 0.682, because low acinar → tumour is itself a good rule.

So an AUC below 0.5 isn't weak discrimination; it's discrimination in the opposite direction. Distance from 0.5 is what measures separability:


| compartment | AUC | abs(AUC − 0.5) | direction | 
|---|---|---|---| 
| Fibroblast | 0.727 | 0.227 | up in tumour | 
| Acinar | 0.318 | 0.182 | **up in normal** | 
| Ductal-1 | 0.334 | 0.166 | up in normal | 
| Ductal-2 | 0.662 | 0.162 | up in tumour | 
| T cell | 0.346 | 0.154 | up in normal | 
| Endocrine | 0.489 | 0.011 | none | 


I compared 0.682 against Ductal-2's 0.662 to make the point that acinar *loss* separates tumour from normal slightly better than malignant *gain* does — which is the wrong ordering if θ_malignant were a good purity estimate.

Though I'd soften that claim now that they're side by side: 0.682 vs 0.662 is a difference of 0.02, well inside the noise at n=15 normals. The honest statement is that they're comparable, and that Fibroblast at 0.227 is the strongest single discriminator — consistent with desmoplasia being the most reliable bulk signature of PDAC.


Because they answer different questions, and I wanted the first.

**|AUC − 0.5|** measures *how much* separation there is, ignoring direction. It's the natural ranking when comparing compartments against each other, since a compartment at 0.32 and one at 0.68 are equally informative — one is depleted in tumours, the other enriched, both equally useful for telling the groups apart.

**1 − AUC** just relabels which class is positive. It's the right transform when you want to state the result in the other direction — as I did for acinar, to compare its 0.682 against Ductal-2's 0.662 on a common footing.

Applying `1 − AUC` uniformly to the whole table would have been wrong: it would turn Fibroblast's 0.727 into 0.273 and make the strongest discriminator look like the weakest.

Applying it *selectively* to sub-0.5 rows is a valid alternative and arguably more readable, since every value would then be ≥0.5 with a direction label. That's the same information as my table, folded differently:

| compartment | AUC (as "separability") | direction |
|---|---|---|
| Fibroblast | 0.727 | up in tumour |
| Acinar | 0.682 | up in normal |
| Ductal-1 | 0.666 | up in normal |
| Ductal-2 | 0.662 | up in tumour |

Same ordering as the |AUC − 0.5| column, since folding at 0.5 and measuring distance from 0.5 are monotonically equivalent. I used the deviation form because it keeps the "0 = useless" baseline visible — Endocrine at 0.011 reads more obviously as noise than 0.511 does.

### Ductal 2 - malignant

In [31]:
# has normal samples
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")
print(Zmal.shape)
Zmal.head(3)

(17540, 153)


,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,T-C3L-01124,T-C3L-00625,T-C3N-03439,...,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-8124,N-TCGA-H6-A45N
ENSG00000000003,30.065,55.029,40.439,32.081,32.77,31.182,33.480,52.445,28.622,41.104,...,NaN,20.901,18.993,NaN,NaN,NaN,39.159,NaN,54.487,NaN
ENSG00000000005,0.035,0.297,0.060,0.007,0.04,0.013,0.031,0.020,0.014,0.045,...,NaN,0.037,0.005,NaN,NaN,NaN,0.041,NaN,0.009,NaN
ENSG00000000419,34.732,36.725,35.302,55.946,36.65,38.623,21.696,37.567,35.662,42.774,...,NaN,29.322,37.812,NaN,NaN,NaN,27.419,NaN,30.108,NaN


In [32]:
ok = np.sum([1 if geneid in ref_new.columns.to_list() else 0 for geneid in df_bulk.index])
ok, len(df_bulk), ref_new.shape[1]

(17540, 27177, 17540)

In [33]:
bt = gene_map.reindex(df_bulk.index)["biotype"]
pd.crosstab(bt, df_bulk.index.isin(ref_new.columns), normalize="index").round(3)

col_0,False,True
biotype,,
lncRNA,0.884,0.116
miRNA,0.997,0.003
protein_coding,0.053,0.947


In [34]:
ref_new.head(3)

,ENSG00000225880,ENSG00000187634,ENSG00000188976,ENSG00000187961,ENSG00000187583,ENSG00000188290,ENSG00000187608,ENSG00000188157,ENSG00000237330,ENSG00000131591,...,ENSG00000167355,ENSG00000184999,ENSG00000235910,ENSG00000157335,ENSG00000257008,ENSG00000166573,ENSG00000268182,ENSG00000254453,ENSG00000226245,ENSG00000213424
cell_state,,,,,,,,,,,,,,,,,,,,,
Fibroblast cell,214.0,1246.0,2337.0,115.0,90.0,4981.0,18665.0,1758.0,11.0,274.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Stellate cell,162.0,112.0,1719.0,52.0,26.0,9730.0,16767.0,1611.0,2.0,172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
Macrophage cell,182.0,35.0,1293.0,66.0,47.0,918.0,19924.0,825.0,9.0,157.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### MalignantCluster

In [35]:
import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

0.35.0


In [36]:
root_mprog_disease = cbio.root_mprog_disease
mal_cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True


mc = pml.MalignantCluster(prism=prism, res=res, 
                          df_bulk=df_bulk, ref=ref_new, 
                          root_mprog_disease = root_mprog_disease,
                          mal_cell_name = mal_cell_name,
                          organ="Pancreas")

mc

### pc_theta_pearson and pc_theta_pearson_raw

**The computation.** Run PCA on the samples × genes matrix, take the first 5 principal components, and correlate each PC's sample scores with `theta_mal` (each sample's malignant fraction). `pcs[:, i]` is one number per sample for PC *i*; `theta_mal.values` is one number per sample. `np.corrcoef(...)[0,1]` pulls the off-diagonal — the Pearson r between them.

You get 5 numbers, one per PC. Each answers: *does this dominant axis of variation track tumour purity?*

**The two versions:**

| | matrix | meaning |
|---|---|---|
| `pc_theta_pearson_raw` | `logx` — log2-CPM before decoupling | how much purity is in the data |
| `pc_theta_pearson` | `Xc` — the matrix you actually cluster | how much purity survives into the analysis |

With `decouple_purity=False` they're the same matrix, so the numbers match — your `[-0.596, -0.205, 0.227, -0.359, -0.158]` versus `[-0.596, -0.205, 0.228, -0.359, -0.159]`. The tiny differences are HVG selection, which happens between the two calls.

With `decouple_purity=True`, `Xc` holds residuals from regressing on `theta_mal`, and residuals are **orthogonal to their regressors by construction**. So `pc_theta_pearson` becomes ~1e-15 — pure floating-point noise. It proves the arithmetic worked, nothing about your data. That's why 0.20.1 added the `_raw` version and the `pc_theta_note`: I originally had you reading a number that can only ever be zero.

**Your actual numbers matter.** PC1 at r = −0.596 means ~36% of the leading component's variance is shared with purity, and PC4 at −0.359 adds more. The sign says low-purity samples score high on PC1. Since `X` is what produced the consensus clustering, the 134-gene tumour axis, and the 6/119 splits, purity is a live confound in all of them.

Which is the concrete reason to run `decouple_purity=True` and compare — with the standing caveat that basal-like PDAC is genuinely lower-purity, so some of that r is biology you'd be deleting.

In [37]:
X, diag = mc.prepare_malignant_matrix(decouple_purity=False, 
                                      keep_genes=mc.program1_panel, drop_pattern=r"^N-")

print("Limited by n_hvg (highly variable genes)\n")
print(X.shape)
X.head(3)

excluded 22/153 samples by keep_samples/drop_pattern
Limited by n_hvg (highly variable genes)

(118, 2000)


,ENSG00000001084,ENSG00000001561,ENSG00000002587,ENSG00000002726,ENSG00000002834,ENSG00000003147,ENSG00000003249,ENSG00000003400,ENSG00000004478,ENSG00000005001,...,ENSG00000274211,ENSG00000275342,ENSG00000275395,ENSG00000275410,ENSG00000276180,ENSG00000277363,ENSG00000277972,ENSG00000278259,ENSG00000278535,ENSG00000278540
T-C3L-02890,6.780,5.736,7.171,5.541,8.460,7.229,3.217,7.004,6.726,7.630,...,6.132,6.505,8.483,5.027,4.268,2.718,4.797,7.108,3.834,7.867
T-C3L-03635,6.269,5.560,7.215,6.130,8.503,8.109,4.257,6.930,7.244,7.363,...,6.472,5.909,6.709,6.371,3.522,3.931,4.802,7.070,4.182,8.450
T-C3L-02701,5.989,7.098,7.601,7.242,8.451,7.274,4.592,7.107,7.048,6.365,...,6.274,5.258,7.830,4.836,4.575,3.923,5.229,7.101,4.260,7.952


### Scores

In [38]:
cmap = {
    "malignant":  "Ductal cell type 2",
    "fibroblast": "Fibroblast cell",     # whatever Peng calls stellate/CAF
    "macrophage": "Macrophage cell",
    "endothelial": "Endothelial cell",
}

X_mal2 = mc.compartment_matrix(cmap["malignant"],  min_share=0.3, min_counts=10)
X_fib2 = mc.compartment_matrix(cmap["fibroblast"], min_share=0.3, min_counts=10)
print(X_mal2.shape, X_fib2.shape)

#---------- convert PROGRAMS to geneid -------------

scores, cov = mc.program_scores(compartment_map=cmap, samples=X.index.to_list())

print("\nPrograms x annotated genes")
print("--------------------------")
print(cov[["compartment","program","n_found","n_total"]])   # n_found should be 0 everywhere

(131, 8585) (129, 10853)

Programs x annotated genes
--------------------------
    compartment                 program  n_found  n_total
0     malignant                   basal       12       12
1     malignant               classical       12       12
2     malignant                     emt        6        6
3     malignant                  prolif        6        6
4     malignant          immune_evasion       12       12
5     malignant    antigen_presentation        6        7
6     malignant  desmoplastic_secretome       10       10
7     malignant    angiogenic_secretome        4        5
8    fibroblast                   myCAF        8        8
9    fibroblast                    iCAF        9        9
10   fibroblast                   apCAF        6        6
11   macrophage                      M1        5        6
12   macrophage                     TAM        7        7
13   macrophage                    SPP1        3        3
14   macrophage        mdsc_suppressive        7  

In [39]:
permutations = 5000
force=False
verbose=True

fname_corr=f"prog_corr_scores_x_malig_n{len(scores.index)}"
df_corr = mc.couple_compartments(scores=scores, fname_corr=fname_corr, cell_name=mc.mal_cell_name, n_perm=permutations, force=force, verbose=verbose)
print(df_corr.shape)
df_corr.head(3)

Table opened ((139, 6)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/deconv/prog_corr_scores_x_malig_n118_n_perm_5000_ctr_theta_True.tsv'
(139, 6)


,program_a,program_b,r,p_perm,n,fdr
0,fibroblast.iCAF,endothelial.activated,0.466,2.000e-04,63,0.003
1,malignant.immune_evasion,fibroblast.iCAF,0.412,2.000e-04,112,0.003
2,malignant.immune_evasion,fibroblast.apCAF,0.414,2.000e-04,112,0.003


In [40]:
print(scores.shape)
S = scores.dropna(axis=1, how="all").copy()
print(S.shape)
S.head(3)


(118, 20)
(118, 20)


,malignant.basal,malignant.classical,malignant.emt,malignant.prolif,malignant.immune_evasion,malignant.antigen_presentation,malignant.desmoplastic_secretome,malignant.angiogenic_secretome,fibroblast.myCAF,fibroblast.iCAF,fibroblast.apCAF,macrophage.M1,macrophage.TAM,macrophage.SPP1,macrophage.mdsc_suppressive,endothelial.tip_angio,endothelial.lymphatic,endothelial.activated,malignant.axis_basal_minus_classical,fibroblast.axis_myCAF_minus_iCAF
T-C3L-00277,-0.136,0.270,-0.011,0.018,-0.356,-1.478,-0.158,-0.772,-0.533,-0.467,-1.016,NaN,NaN,NaN,NaN,0.998,-0.347,-1.684,-0.406,-0.066
T-C3L-00589,-0.203,0.006,0.206,-0.697,-0.302,-0.211,-0.120,-0.988,-0.263,-0.059,-0.745,-0.891,0.095,-0.64,-0.655,-0.772,-0.433,-1.188,-0.209,-0.205
T-C3L-00625,-0.034,0.441,-0.184,-0.051,-0.340,-1.349,-0.066,-0.002,-0.233,-0.112,-0.542,NaN,NaN,NaN,NaN,1.098,0.404,-1.042,-0.475,-0.121


### axis_modality: is each axis one cloud or two? BIC of a 1- vs 2-component Gaussian.

**Decisive for interpretation.**

If an axis is unimodal, states carved from it are tails of a continuum, not phenotypes: 
- valid as a stratification,
- but you cannot call them subtypes, 
- and a sample near the boundary has no stable assignment.

### delta_bic

> delta_bic = BIC(1 comp) - BIC(2 comp). 

- positive favours two components; the usual reading is:
  - \>10 strong,
  - 2-10 weak, 
  - <2 negligible.
  
`min_weight` guards the case where the 2-component fit wins only by devoting a tiny component to outliers.

In [41]:
mc.axis_modality(scores)

,axis,n,delta_bic,min_component_weight,separation_sd,skew,verdict
0,malignant.axis_basal_minus_classical,118,16.7,0.225,1.72,1.38,bimodal
1,fibroblast.axis_myCAF_minus_iCAF,112,19.9,0.071,5.16,-1.03,weak/outlier-driven


In [42]:
coh = pd.Series(np.where(scores.index.str.contains("TCGA"), "TCGA", "CPTAC"),  index=scores.index)

for name in ["TCGA", "CPTAC"]:
    idx = list(scores.index[coh == name])
    sc_c, _ = mc.program_scores(compartment_map=cmap, samples=idx)
    print(name, len(idx))
    print(mc.axis_modality(sc_c).to_string(index=False))
    print("\n----------------------------------\n")

TCGA 74
                                axis  n  delta_bic  min_component_weight  separation_sd  skew             verdict
malignant.axis_basal_minus_classical 74        6.7                 0.027           6.15  1.53 weak/outlier-driven
    fibroblast.axis_myCAF_minus_iCAF 68       11.2                 0.153           3.42 -1.11             bimodal

----------------------------------

CPTAC 44
                                axis  n  delta_bic  min_component_weight  separation_sd  skew  verdict
malignant.axis_basal_minus_classical 44       -1.5                 0.194           2.74  0.99 unimodal
    fibroblast.axis_myCAF_minus_iCAF 44      -11.1                 0.488           1.64 -0.42 unimodal

----------------------------------



### "prolif" x "iCAF"

In [43]:
force=False
verbose=False

for name in ["TCGA","CPTAC"]:
    idx = list(scores.index[coh==name])
    sc_c,_ = mc.program_scores(compartment_map=cmap, samples=idx)

    fname_corr=f"prog_corr_scores_x_malig_for_{name}_n{len(sc_c.index)}"
    R = mc.couple_compartments(scores=sc_c, fname_corr=fname_corr, cell_name=mc.mal_cell_name, n_perm=permutations, force=force, verbose=verbose)

    # and "_minus" not in x.program_b
    print(name, '\n', R[R.apply(lambda x: "prolif" in x.program_a+x.program_b
                          and "iCAF" in x.program_a+x.program_b, axis=1)][["program_a", "program_b", "n", "r","p_perm","fdr"]].to_string(index=False))

TCGA 
        program_a                        program_b  n      r    p_perm   fdr
malignant.prolif                  fibroblast.iCAF 68 -0.458 2.000e-04 0.014
malignant.prolif fibroblast.axis_myCAF_minus_iCAF 68  0.398 9.998e-04 0.035
CPTAC 
        program_a                        program_b  n      r    p_perm   fdr
malignant.prolif                  fibroblast.iCAF 44 -0.505 9.998e-04 0.035
malignant.prolif fibroblast.axis_myCAF_minus_iCAF 44  0.175 2.601e-01 0.657


The core coupling replicates cleanly and is now robust to everything we've thrown at it:

| | r | FDR |
|---|---|---|
| TCGA (n=74) | **−0.458** | 0.023 |
| CPTAC (n=44) | **−0.505** | 0.035 |

**Same sign, near-identical magnitude, independently significant, θ-controlled, and now on a corrected reference with ~990 recovered genes, a rebuilt marker set and ENSG identifiers. That's a genuine finding: malignant proliferation co-occurs with iCAF-depleted stroma.**

The second row in each block is the same relationship read on the `myCAF_minus_iCAF` contrast (positive, since iCAF is the negative pole). It's significant in TCGA (r=0.398) but not CPTAC (r=0.175, FDR 0.646) — so report the direct `prolif`↔`iCAF` pair, not the axis version. The axis adds myCAF variation as noise without adding signal.

Worth noting *why* this one survived when everything else didn't. It's a **negative** correlation, and the shared-bulk component that inflates `zm`/`zf` to r≈0.93 is positive by construction. An artifact from that source can't produce a negative coupling. That's a structural argument for its validity, independent of the replication.

Two things I'd still do before writing it up.

Check it isn't purity in disguise — `couple_compartments` controls θ by default, but a direct look is cheap:

And be explicit that it's correlational and undirected. Bulk data cannot say whether proliferative tumours suppress iCAFs, iCAFs restrain proliferation, or both follow from something else. The mechanistic literature would let you argue either way — which is exactly why the claim should stay at co-occurrence.

In [44]:
force=False
verbose=False

for name in ["TCGA","CPTAC"]:
    idx = list(scores.index[coh==name])
    sc_c,_ = mc.program_scores(compartment_map=cmap, samples=idx)

    fname_corr=f"prog_corr_scores_x_malig_for_{name}_n{len(sc_c.index)}"
    R = mc.couple_compartments(scores=sc_c, fname_corr=fname_corr, cell_name=mc.mal_cell_name, control_theta=False, n_perm=permutations, force=force, verbose=verbose)
    if name == 'TCGA':
        R1 = R.copy()
    else:
        R2 = R.copy()

    # and '_minus_' not in x.program_b
    print(name, "\n", R[R.apply(lambda x: "prolif" in x.program_a+x.program_b
                          and x.program_b.endswith("iCAF"), axis=1)][["program_a", "program_b", "n", "r","fdr"]].to_string(index=False))

TCGA 
        program_a                        program_b  n      r   fdr
malignant.prolif                  fibroblast.iCAF 68 -0.462 0.028
malignant.prolif fibroblast.axis_myCAF_minus_iCAF 68  0.428 0.046
CPTAC 
        program_a                        program_b  n      r   fdr
malignant.prolif                  fibroblast.iCAF 44 -0.521 0.028
malignant.prolif fibroblast.axis_myCAF_minus_iCAF 44  0.171 0.692


In [45]:
for name in ["TCGA","CPTAC"]:
    idx = list(scores.index[coh==name])
    sc_c,_ = mc.program_scores(compartment_map=cmap, samples=idx)
    
    x, y = sc_c["malignant.prolif"], sc_c["fibroblast.iCAF"]

    ok = x.notna() & y.notna()
    boot = [spearmanr(*zip(*[(x[ok].iloc[i], y[ok].iloc[i])
            for i in np.random.default_rng(s).integers(0, ok.sum(), ok.sum())]))[0]
            for s in range(500)]
    
    print(f"{name:6} r={spearmanr(x[ok],y[ok])[0]:.3f}  95% CI [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]")


TCGA   r=-0.462  95% CI [-0.655, -0.216]
CPTAC  r=-0.521  95% CI [-0.693, -0.277]


Both CIs exclude zero comfortably and overlap heavily — that's the cleanest evidence you've had for anything in this study.

| cohort | r | 95% CI |
|---|---|---|
| TCGA (n=74) | −0.462 | [−0.655, −0.216] |
| CPTAC (n=44) | −0.521 | [−0.693, −0.277] |

Two independent estimates, non-overlapping datasets, consistent point estimates. Lead with these rather than FDR — a bootstrap CI is the honest summary when the test was one of ~340, and the replication carries the inference regardless of which correction you quote.

Robust to: cohort, reference version (pre/post ENSG harmonisation, ~990 recovered genes, rebuilt marker set), θ control, and identifier scheme. Negative sign, so not producible by the shared-bulk component.

The result: **malignant proliferation is inversely associated with inflammatory CAF content in deconvolved PDAC transcriptomes.**

Three caveats to carry into the writeup, none of which undermine it:

- **Correlational and undirected.** Bulk cannot distinguish proliferative tumours suppressing iCAFs from iCAFs restraining proliferation, or a common upstream cause.
- **One reference.** Every compartment comes from a single Peng-2019 fit; a reference misspecification propagates to all of them, and nothing internal detects it. A second reference — snRNA-seq, which would also raise your 11.6% lncRNA ceiling — would be the strongest possible validation.
- **Compartment separation is partial.** `zm` vs `zf` at r ≈ 0.93 means most of each matrix is shared. The contrast structure of program scores is what makes this measurable, and it's worth saying so explicitly.

Alongside the replicated negative on discrete subtypes, that's a coherent paper: two continuous compartment axes, no discrete subtypes in either cohort, and one reproducible tumour–stroma association.


### compare_coupled_compartments()

In [46]:
program_list = ['TCGA', 'CPTAC']

df_meta_coup = mc.compare_coupled_compartments(program_list=program_list, coh=coh, cmap=cmap, 
                                          scores=scores, n_perm=permutations, force=force, verbose=verbose)
print(df_meta_coup.shape)
df_meta_coup.head(3)

(139, 18)


,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,z_het,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant
0,malignant.antigen_presentation,fibroblast.apCAF,0.486,2.000e-04,68,0.014,0.167,2.795e-01,44,0.673,1.819,0.069,0.391,0.372,5.747e-05,3.995e-03,0.698,True
1,malignant.prolif,fibroblast.iCAF,-0.458,2.000e-04,68,0.014,-0.505,9.998e-04,44,0.035,0.308,0.758,-0.518,-0.476,9.518e-08,1.323e-05,0.000,True
2,malignant.immune_evasion,fibroblast.iCAF,0.425,3.999e-04,68,0.019,0.212,1.698e-01,44,0.576,1.196,0.232,0.361,0.346,1.994e-04,9.240e-03,0.301,True


In [47]:
df_meta_ok = df_meta_coup[(df_meta_coup.fdr_meta < 0.10) & (df_meta_coup.r_meta.abs() >= 0.25) & df_meta_coup.concordant]
df_meta_ok = df_meta_ok.sort_values(['program_a', 'program_b'])
df_meta_ok

,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,z_het,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant
64,fibroblast.apCAF,endothelial.activated,0.152,3.757e-01,35,0.798,0.607,7.998e-04,28,0.035,-2.064,0.039,0.395,0.376,2.869e-03,4.431e-02,0.765,True
38,fibroblast.iCAF,endothelial.activated,0.253,1.432e-01,35,0.510,0.635,2.000e-04,28,0.014,-1.844,0.065,0.474,0.442,3.433e-04,1.193e-02,0.706,True
10,malignant.angiogenic_secretome,macrophage.TAM,-0.473,7.798e-03,32,0.099,-0.229,2.314e-01,29,0.657,-1.041,0.298,-0.381,-0.364,4.728e-03,5.975e-02,0.077,True
0,malignant.antigen_presentation,fibroblast.apCAF,0.486,2.000e-04,68,0.014,0.167,2.795e-01,44,0.673,1.819,0.069,0.391,0.372,5.747e-05,3.995e-03,0.698,True
108,malignant.antigen_presentation,macrophage.M1,0.050,7.788e-01,32,0.974,0.663,2.000e-04,29,0.014,-2.767,0.006,0.403,0.383,2.770e-03,4.431e-02,0.869,True
15,malignant.desmoplastic_secretome,endothelial.lymphatic,-0.373,2.500e-02,36,0.196,-0.396,3.959e-02,28,0.393,0.102,0.919,-0.404,-0.383,2.102e-03,4.174e-02,0.000,True
6,malignant.immune_evasion,endothelial.activated,0.459,5.399e-03,36,0.096,0.373,5.139e-02,28,0.453,0.394,0.694,0.451,0.423,5.943e-04,1.652e-02,0.000,True
9,malignant.immune_evasion,fibroblast.apCAF,0.338,7.199e-03,68,0.099,0.178,2.428e-01,44,0.657,0.861,0.389,0.285,0.278,3.298e-03,4.584e-02,0.000,True
2,malignant.immune_evasion,fibroblast.iCAF,0.425,3.999e-04,68,0.019,0.212,1.698e-01,44,0.576,1.196,0.232,0.361,0.346,1.994e-04,9.240e-03,0.301,True
3,malignant.prolif,fibroblast.axis_myCAF_minus_iCAF,0.398,9.998e-04,68,0.035,0.175,2.601e-01,44,0.657,1.225,0.221,0.326,0.315,7.774e-04,1.801e-02,0.334,True


In [48]:
df_meta_ok = df_meta_coup[(df_meta_coup.p_perm_t < 0.05) & (df_meta_coup.p_perm_c < 0.05) &
                     df_meta_coup.concordant & (df_meta_coup.r_meta.abs() >= 0.25)]
df_meta_ok = df_meta_ok.sort_values(['program_a', 'program_b'])
df_meta_ok

,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,z_het,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant
15,malignant.desmoplastic_secretome,endothelial.lymphatic,-0.373,2.500e-02,36,0.196,-0.396,3.959e-02,28,0.393,0.102,0.919,-0.404,-0.383,2.102e-03,4.174e-02,0.0,True
1,malignant.prolif,fibroblast.iCAF,-0.458,2.000e-04,68,0.014,-0.505,9.998e-04,44,0.035,0.308,0.758,-0.518,-0.476,9.518e-08,1.323e-05,0.0,True


### Coupled & NNLS non-prior biased

In [49]:
th_nnls = prism.nnls_deconvolve(df_bulk, ref_new, genes=gene_subset)

th_em   = res.theta

common = th_em.index.intersection(th_nnls.index)
df_conc = pd.DataFrame({
    "nnls_spearman":  {c: th_em.loc[common, c].corr(th_nnls.loc[common, c], method="spearman")
                  for c in th_em.columns},
    "nnls_pearson":   {c: th_em.loc[common, c].corr(th_nnls.loc[common, c])
                  for c in th_em.columns},
    "nnls_mean_em":   th_em.loc[common].mean(),
    "nnls_mean_nnls": th_nnls.loc[common].mean(),
    "nnls_n":         {c: (th_em.loc[common, c].notna() & th_nnls.loc[common, c].notna()).sum()
                  for c in th_em.columns},
}).sort_values("nnls_spearman", ascending=False).round(3)

# each coupling has two compartments -> map both, gate on the weaker one
cell_of = {k: v for k, v in cmap.items()}            # program key -> cell type
sp = df_conc["nnls_spearman"]

def conc_of(prog):                                   # "malignant.prolif" -> spearman
    return sp.get(cell_of.get(prog.split(".")[0]), np.nan)

df_meta_coup["nnls_prog_a"]   = df_meta_coup.program_a.map(conc_of)
df_meta_coup["nnls_prog_b"]   = df_meta_coup.program_b.map(conc_of)
df_meta_coup["nnls_min"] = df_meta_coup[["nnls_prog_a", "nnls_prog_b"]].min(axis=1)

df_meta_coup.head(3)


,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,...,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant,nnls_prog_a,nnls_prog_b,nnls_min
0,malignant.antigen_presentation,fibroblast.apCAF,0.486,2.000e-04,68,0.014,0.167,2.795e-01,44,0.673,...,0.069,0.391,0.372,5.747e-05,3.995e-03,0.698,True,0.864,0.932,0.864
1,malignant.prolif,fibroblast.iCAF,-0.458,2.000e-04,68,0.014,-0.505,9.998e-04,44,0.035,...,0.758,-0.518,-0.476,9.518e-08,1.323e-05,0.000,True,0.864,0.932,0.864
2,malignant.immune_evasion,fibroblast.iCAF,0.425,3.999e-04,68,0.019,0.212,1.698e-01,44,0.576,...,0.232,0.361,0.346,1.994e-04,9.240e-03,0.301,True,0.864,0.932,0.864


In [50]:
df_meta_ok = df_meta_coup[
    (df_meta_coup.p_perm_t < 0.1) & (df_meta_coup.p_perm_c < 0.1) &   # both cohorts, not just meta
     df_meta_coup.concordant &
    (df_meta_coup.r_meta.abs() >= 0.2) &
    (df_meta_coup.nnls_min > 0.2)
].sort_values("fdr_meta")

df_meta_ok

,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,...,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant,nnls_prog_a,nnls_prog_b,nnls_min
1,malignant.prolif,fibroblast.iCAF,-0.458,2.000e-04,68,0.014,-0.505,9.998e-04,44,0.035,...,0.758,-0.518,-0.476,9.518e-08,1.323e-05,0.0,True,0.864,0.932,0.864


### only malignant.prolif    fibroblast.iCAF survives

That's the right outcome, and the filters earned it — each one removed a distinct failure mode rather than just tightening a threshold.

What `prolif`↔`iCAF` has now survived:

- both cohorts independently significant (not meta-only)
- bootstrap CIs excluding zero: TCGA [−0.655, −0.216], CPTAC [−0.693, −0.277]
- I² = 0, p_meta 9.5e-8
- two reference versions, pre and post ENSG harmonisation
- θ control on and off
- acinar content (residual normal tissue)
- both compartments identified by the NNLS cross-check
- negative sign, which the shared-bulk component can't produce

One result from ~340 tested pairs, but arrived at through replication rather than selection — which is the distinction that matters.

Worth being explicit in the writeup that this is what a null-dominated screen looks like when it's run honestly. Six candidates passed a naive meta-FDR and none survived replication plus compartment-identifiability. That's a feature of the analysis, not a disappointment, and stating it makes the surviving result more credible rather than less.

Two things I'd still recommend, in order of cost:

**Merge D1/D2 and re-test.** Cheap, and it would let you phrase the finding on a compartment whose fractions are defensible rather than one whose split isn't identified.

**A second reference as validation.** Chan-Seng-Yue 2020 uses CNV rather than expression to call malignant cells, so it's orthogonal on exactly the axis where Peng is weakest. If the coupling reproduces there, the single-reference limitation — currently the softest point in your writeup — largely goes away.

In [51]:
hi = (scores["malignant.prolif"] > scores["malignant.prolif"].median()) & \
     (scores["fibroblast.iCAF"] < scores["fibroblast.iCAF"].median())
pd.crosstab(hi, coh.reindex(scores.index))

col_0,CPTAC,TCGA
row_0,,
False,30,52
True,14,22


### Malignat proliferation x iCAF

In [52]:
th_nnls = prism.nnls_deconvolve(df_bulk, ref_new, genes=gene_subset)
print(th_nnls.shape)
th_nnls.head(3)

(153, 10)


cell_state,Fibroblast cell,Stellate cell,Macrophage cell,Endothelial cell,T cell,B cell,Ductal cell type 2,Endocrine cell,Ductal cell type 1,Acinar cell
T-C3L-02890,0.832,0.051,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.117
T-C3L-03635,0.954,0.046,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000
T-C3L-02701,0.917,0.083,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000


In [53]:
th_em   = res.theta

common = th_em.index.intersection(th_nnls.index)

df_conc = pd.DataFrame({
    "spearman":  {c: th_em.loc[common, c].corr(th_nnls.loc[common, c], method="spearman")
                  for c in th_em.columns},
    "pearson":   {c: th_em.loc[common, c].corr(th_nnls.loc[common, c])
                  for c in th_em.columns},
    "mean_em":   th_em.loc[common].mean(),
    "mean_nnls": th_nnls.loc[common].mean(),
    "n":         {c: (th_em.loc[common, c].notna() & th_nnls.loc[common, c].notna()).sum()
                  for c in th_em.columns},
}).sort_values("spearman", ascending=False).round(3)


df_conc

,spearman,pearson,mean_em,mean_nnls,n
Acinar cell,0.960,0.994,0.159,0.210,145
Fibroblast cell,0.932,0.891,0.370,0.572,145
Ductal cell type 2,0.864,0.755,0.340,0.087,145
Ductal cell type 1,0.570,0.151,0.021,0.032,145
Stellate cell,0.431,0.695,0.026,0.069,145
Endocrine cell,0.319,0.458,0.028,0.002,145
Endothelial cell,0.194,0.182,0.022,0.004,145
B cell,0.166,0.043,0.008,0.004,145
T cell,0.129,0.090,0.001,0.000,145
Macrophage cell,0.116,0.067,0.026,0.019,145


### Conclusion

Your current numbers include Endocrine 0.32, Stellate 0.43 and Ductal-1 0.57, which fill exactly the gap I claimed didn't exist. The distribution is continuous, not bimodal, and my "two obvious groups" reading was an artifact of quoting stale values.

That makes the threshold question harder rather than easier. There's no natural break to cut at, so any cutoff is a judgement call — and Ductal-1 at 0.57 is precisely the awkward case: too high to dismiss, too low to trust, and it happens to be the compartment implicated in the D1/D2 identifiability problem the AUC test exposed.

The honest treatment is a graded one rather than a binary:

| Spearman | compartments | reading |
|---|---|---|
| >0.85 | Acinar, Fibroblast, Ductal-2 | data-driven |
| 0.3–0.6 | Ductal-1, Stellate, Endocrine | partly prior-driven; interpret cautiously |
| <0.2 | T cell, macrophage, B cell, endothelial | not identified |

And report the values rather than a pass/fail, so a reader can apply their own standard.

Two things worth adding for the middle band, since that's where judgement is doing the work. Ductal-1 and Ductal-2 both being in play, with the AUC test showing D1+D2 separates better than either alone, suggests their agreement is partly *shared* rather than independent — the estimators may agree on the total while splitting it differently. Worth checking:

```python
d12_em   = th_em[["Ductal cell type 1","Ductal cell type 2"]].sum(axis=1)
d12_nnls = th_nnls[["Ductal cell type 1","Ductal cell type 2"]].sum(axis=1)
print("D1+D2 spearman:", d12_em.corr(d12_nnls, method="spearman").round(3))
```

If that comes back above both individual values, the total is identified and the split isn't — which is a cleaner statement than any per-compartment threshold.

In [54]:
d12_em   = th_em[["Ductal cell type 1","Ductal cell type 2"]].sum(axis=1)
d12_nnls = th_nnls[["Ductal cell type 1","Ductal cell type 2"]].sum(axis=1)
print("D1+D2 spearman:", d12_em.corr(d12_nnls, method="spearman").round(3))

D1+D2 spearman: 0.83


That's a clean result, and it changes what you can say.

**D1+D2 = 0.83**, against D1 = 0.57 and D2 = 0.86 individually. Combining them lifts the weak compartment substantially, and it matches the AUC pattern exactly (D1+D2 = 0.74 vs D2 = 0.66 for tumour/normal separation). Two independent tests, same conclusion:

#### **Total ductal epithelium is identified. The split into normal-like and malignant ductal is not.**

That's a much more precise statement than "Ductal-1 is prior-prone." The estimators and the ground-truth test both agree on how much ductal epithelium a sample has; they disagree on how to divide it, because Peng's D1 and D2 reference profiles are too similar to separate at bulk resolution.

One asymmetry worth noting: D2 alone (0.86) slightly exceeds the sum (0.83), which is a bit odd if the split were pure noise. D2 is the larger compartment (mean θ ~0.35 vs ~0.02), so it dominates the sum — the sum's agreement is mostly D2's, and D1's contribution adds noise. Meaning D1 is the unidentified one specifically, rather than the split being symmetrically ambiguous.

For the writeup, this supports a defensible framing: report the ductal compartment as **ductal epithelium** where possible, note that D1/D2 assignment is not reliably identified, and phrase your finding as a proliferation program in the ductal epithelial compartment rather than in malignant cells.

It also suggests a concrete option if you ever revisit the deconvolution: merge D1 and D2 into one cell type in `state_to_type` before fitting. You'd lose the malignant/normal distinction — which you don't reliably have anyway — and gain a compartment that's actually identified. Worth testing whether `prolif`↔`iCAF` survives that, since it would then rest on a compartment with defensible fractions.

### Perhaps chosing another study different from Peng's we will get a better result (?)

Plausibly, but I'd be cautious — the reference may not be the binding constraint.

**Where a new reference clearly helps.** The lncRNA ceiling is a chemistry problem: 11.6% coverage because 10x 3' under-captures them. A snRNA-seq or total-RNA reference would raise that substantially, and that's a real gain for your Program-1 work.

**Where it may not.** The D1/D2 confusion is not obviously Peng's fault. Normal ductal and malignant ductal cells are transcriptionally similar — that's the biology, not the annotation. A different study will still label two ductal populations whose bulk-level profiles overlap. Steele 2020 and Lin 2020 would face the same problem.

Chan-Seng-Yue 2020 is the interesting exception, because it uses **whole-genome CNV** to call malignant cells rather than expression clustering. That's an orthogonal criterion, so its malignant/non-malignant boundary isn't circular with expression the way Peng's is. It's also snRNA-seq, so it addresses the lncRNA ceiling at the same time.

#### Two cautions before committing.

Swapping references invalidates everything and costs a full re-run — and you've just done that once. The cheaper test is to **merge D1 and D2** in `state_to_type` and see whether `prolif`↔`iCAF` survives on an identified compartment. That's one afternoon, not a week, and it tells you whether the ductal split matters to your result at all.

And the strongest use of a second reference isn't replacement but **validation**. If `prolif`↔`iCAF` reproduces under Chan-Seng-Yue as well as under Peng, that addresses the "single reference" limitation directly — which is currently the weakest point in your writeup, since a reference misspecification propagates to every compartment and nothing internal detects it.

So the order I'd suggest: merge D1/D2 first (cheap, answers whether the split matters), then a second reference as validation rather than as a fix.

### Cluster with program's expression signatures

In [55]:
df_meta_coup.head(2)

,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,...,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant,nnls_prog_a,nnls_prog_b,nnls_min
0,malignant.antigen_presentation,fibroblast.apCAF,0.486,2.000e-04,68,0.014,0.167,2.795e-01,44,0.673,...,0.069,0.391,0.372,5.747e-05,3.995e-03,0.698,True,0.864,0.932,0.864
1,malignant.prolif,fibroblast.iCAF,-0.458,2.000e-04,68,0.014,-0.505,9.998e-04,44,0.035,...,0.758,-0.518,-0.476,9.518e-08,1.323e-05,0.000,True,0.864,0.932,0.864


In [56]:
df_meta_ok = df_meta_coup[(df_meta_coup.p_perm_t < 0.05) & (df_meta_coup.p_perm_c < 0.05) &
                     df_meta_coup.concordant & (df_meta_coup.r_meta.abs() >= 0.3) & 
                     (df_meta_coup.nnls_prog_a.abs() >= 0.3) & (df_meta_coup.nnls_prog_b.abs() >= 0.3)]
df_meta_ok = df_meta_ok.sort_values(['program_a', 'program_b'])
df_meta_ok

,program_a,program_b,r_t,p_perm_t,n_t,fdr_t,r_c,p_perm_c,n_c,fdr_c,...,p_het,z_meta,r_meta,p_meta,fdr_meta,I2,concordant,nnls_prog_a,nnls_prog_b,nnls_min
1,malignant.prolif,fibroblast.iCAF,-0.458,2.000e-04,68,0.014,-0.505,9.998e-04,44,0.035,...,0.758,-0.518,-0.476,9.518e-08,1.323e-05,0.0,True,0.864,0.932,0.864


In [57]:
# only tumors
keep = list(scores.index)
dic_mats = {k: mc.compartment_matrix(v, samples=keep, min_share=0.3, min_counts=10)
            for k, v in cmap.items()}

progs = sorted(set(df_meta_ok.program_a) | set(df_meta_ok.program_b))

blocks = []
for program in progs:
    print(program)
    comp, name = program.split(".", 1)
    if name.startswith("axis_"):        # axes have no genes of their own
        continue
    df_map = dic_mats[comp]
    # print(df_map)
    geneid_list = [x for x in mc.PROGRAMS_ENSG[comp][name] if x in df_map.columns]
    df_z = (df_map[geneid_list] - df_map[geneid_list].mean()) / df_map[geneid_list].std().replace(0, np.nan)
    df_z.columns = [f"{program}|{col}" for col in df_z.columns]

    blocks.append(df_z)
    print("")

common = sorted(set.intersection(*(set(b.index) for b in blocks)))
df_gene = pd.concat([b.loc[common] for b in blocks], axis=1)
df_gene = df_gene.dropna(axis=1, how="any")
print(df_gene.shape)

fibroblast.iCAF

malignant.prolif

(112, 15)


In [58]:
df_gene.head(10).T

,T-C3L-00277,T-C3L-00589,T-C3L-00625,T-C3L-00640,T-C3L-00819,T-C3L-00881,T-C3L-01032,T-C3L-01051,T-C3L-01124,T-C3L-01598
fibroblast.iCAF|ENSG00000136244,-0.536,-1.061,-0.016,-0.013,0.239,-0.280,-1.152,-1.383,-1.017,-0.065
fibroblast.iCAF|ENSG00000107562,-0.193,-0.804,-0.196,0.125,-1.148,0.147,-0.622,-1.449,-0.164,-1.102
fibroblast.iCAF|ENSG00000134853,0.185,1.084,0.963,-0.253,-0.215,0.973,0.172,1.055,1.538,0.586
fibroblast.iCAF|ENSG00000105509,-1.061,-0.541,-0.770,-0.017,-0.199,-0.460,-0.933,-1.189,-0.886,-0.757
fibroblast.iCAF|ENSG00000170961,-0.090,-0.357,-0.808,-0.336,-0.374,-0.019,-0.864,-0.733,-0.244,-0.957
fibroblast.iCAF|ENSG00000197766,-0.372,1.451,-0.278,0.592,-0.219,0.197,0.112,-0.507,-1.380,1.020
fibroblast.iCAF|ENSG00000160789,-0.579,0.265,-0.763,0.746,-0.152,-1.444,-0.060,-0.910,-1.195,-1.018
fibroblast.iCAF|ENSG00000125730,-2.115,-0.700,0.605,0.101,-1.098,0.797,0.612,-0.791,0.302,0.718
fibroblast.iCAF|ENSG00000145824,0.532,0.180,0.258,-0.310,-0.290,0.360,0.103,0.106,-0.201,0.460
malignant.prolif|ENSG00000148773,0.942,-0.158,0.849,0.436,0.215,-0.343,1.364,-0.280,0.837,-0.090


In [59]:
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

pca = PCA(5).fit(df_gene - df_gene.mean())
P = pd.DataFrame(pca.transform(df_gene - df_gene.mean()), index=df_gene.index,
                 columns=[f"PC{i+1}" for i in range(5)])
print("Explained variance:", pca.explained_variance_ratio_.round(3))
print()

cohG = coh.reindex(P.index)
for c in ["PC1","PC2","PC3"]:
    print(c, "cohort AUC", round(roc_auc_score((cohG=="TCGA").astype(int), P[c]), 3))

print()

# is PC1 just the coupling?
print("PC1 ~ prolif :", round(P.PC1.corr(scores.loc[P.index,"malignant.prolif"], method="spearman"), 3))
print("PC1 ~ iCAF   :", round(P.PC1.corr(scores.loc[P.index,"fibroblast.iCAF"],  method="spearman"), 3))

print()

print("PC2 ~ prolif :", round(P.PC2.corr(scores.loc[P.index,"malignant.prolif"], method="spearman"), 3))
print("PC2 ~ iCAF   :", round(P.PC2.corr(scores.loc[P.index,"fibroblast.iCAF"],  method="spearman"), 3))

print()

mc.axis_modality(P[["PC1","PC2"]].rename(columns=lambda c: f"pc.axis_{c}"))

Explained variance: [0.342 0.176 0.149 0.08  0.056]

PC1 cohort AUC 0.538
PC2 cohort AUC 0.588
PC3 cohort AUC 0.91

PC1 ~ prolif : -0.832
PC1 ~ iCAF   : 0.834

PC2 ~ prolif : 0.494
PC2 ~ iCAF   : 0.324



,axis,n,delta_bic,min_component_weight,separation_sd,skew,verdict
0,pc.axis_PC1,112,-5.7,0.280,1.49,0.61,unimodal
1,pc.axis_PC2,112,-7.7,0.466,1.73,0.52,unimodal


### Cluster conclusions

PC2 completes the picture, and it's the informative addition.

PC1 loads prolif **negatively** and iCAF **positively** (−0.83, +0.83) — the contrast, i.e. your coupling. PC2 loads both **positively** (+0.49, +0.32) — the sum, i.e. shared variation across both compartments. That's the classic contrast/sum decomposition, and PCA found it unprompted.

Two readings worth taking from that.

**PC2 is where the shared-bulk component lives.** Recall `zm` vs `zf` at r≈0.93 — most of each compartment matrix is common.

- **PC1 at 34% captures the part where the two compartments genuinely diverge**; 
- **PC2 at 18% captures where they move together.** 
- The fact that your finding sits on the contrast axis and not the sum axis is another structural argument for it: shared-bulk artifacts load on PC2, not PC1.

**Neither is bimodal**, and both ΔBIC values are negative — one component preferred outright. Combined with the per-cohort GMM and the consensus clustering, that's three independent representations agreeing there are no discrete subtypes.

The one thing I'd flag: **PC3 at cohort AUC 0.91** is a strong batch axis holding 15% of variance. It stayed out of PC1/PC2 here, but with only 15 genes the variance structure is simple. If you widen to 60–80 genes across more programs, that cohort signal could easily rise — and if it lands in PC1 or PC2, it would mean the multi-program gene space is batch-dominated and only the contrast-based scores are usable.

Worth running that wider basis once, purely to know where the cohort axis ends up. It's the last thing I'd check before writing this section.

### Descriptive

In [60]:

disc = mc.discretize_axes(scores)

print("in malignant", end=" ")
R_mal = mc.factorial_state_de(X_mal2, disc)
print("in fibroblast", end=" ")
R_fib = mc.factorial_state_de(X_fib2, disc)

print("-----------"*5)

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)   # the 1604
hits = R_fib.index[R_fib.filter(like="fdr_B_").iloc[:,0] < 0.05]
print(f"{len(set(hits) & fitted)}/{len(hits)} hits are fitted genes")
print(f"background: {len(fitted & set(R_fib.index))}/{len(R_fib)}")

print("-----------"*5)

common_genes = X_mal2.columns.intersection(X_fib2.columns)
Z_mal_common = ((X_mal2[common_genes] - X_mal2[common_genes].mean()) / X_mal2[common_genes].std())
Z_fib_common = ((X_fib2[common_genes] - X_fib2[common_genes].mean()) / X_fib2[common_genes].std())
Z_corr = pd.Series({gene: Z_mal_common[gene].corr(Z_fib_common[gene]) for gene in common_genes})

fitted = set(res.cell_type_expression(cmap["fibroblast"]).index)
print("fitted    genes, median r:", Z_corr[ Z_corr.index.isin(fitted)].median().round(3))
print("projected genes, median r:", Z_corr[~Z_corr.index.isin(fitted)].median().round(3))

in malignant excluded 0 program marker genes; 8585 remain
in fibroblast excluded 0 program marker genes; 10853 remain
-------------------------------------------------------
62/194 hits are fitted genes
background: 696/10853
-------------------------------------------------------
fitted    genes, median r: 0.912
projected genes, median r: 0.898


In [61]:
disc.head(3)

,basal_minus_classical,myCAF_minus_iCAF
T-C3L-00277,low,low
T-C3L-00589,low,low
T-C3L-00625,low,low


## Tahoe

In [ ]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]
        # min probes per block

In [ ]:
cov["block_probe_counts"]

In [ ]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

In [ ]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

In [ ]:
cov["missing_symbol"]

### Shards - A database shard, or simply a shard, is a horizontal partition of data within a database or search engine.

In [ ]:
cvcl = mc.organ_cell_lines()
print(f"{len(cvcl)} -", "; ".join(cvcl)) 

In [ ]:
# The sampling interval — read the footer of every 6th shard instead of all 1026.
idx = mc.shard_index(stride=6)          # ~120 new footers, ~9 min (not 172)

shards = mc.find_shards_for(cvcl) # expect: located 11, missing 0

In [ ]:
shards

In [ ]:
dropped = ('CVCL_0186','CVCL_1634','CVCL_1638','CVCL_1639')
cvcl7  = [c for c in cvcl if c not in dropped]
shards = mc.find_shards_for(cvcl7)          # expect missing: 0

In [ ]:
mc.shard_sizes(shards)

In [ ]:
mc.shard_sizes(shards)["bytes"].sum() / 1e9      # GB for the 237 shards

### transform to geneid

But genes=X.columns is now ENSG, and Tahoe's gene_name column holds HGNC symbols. That query would return zero rows. Translate first:

In [ ]:
# to much: lets paralelize
# mc.download_shards(shards, dry_run=True)
mc.download_shards(shards, max_workers=8)

In [ ]:
gene_map.head(3)

In [ ]:
"""
df["condition"] = df["cell_line_id"].astype(str) + "|" + df["drug"].astype(str)
df_pivot = (df.pivot(index="gene", columns="condition", values="stat")
        .astype(dtype))
"""

force=True


id2sym = gene_map["symbol"].astype(str).to_dict()
genes_sym = [id2sym[g] for g in X.columns if g in id2sym]
print(len(genes_sym), "of", X.shape[1])

df_pivot, cond = mc.load_tahoe_de(genes=genes_sym, organs=("Pancreas",),
                                  mode="download", _shard_subset=shards, force=force)

print(df_pivot.shape)                                  # genes x conditions
df_pivot.head(8)

In [ ]:
print(df_pivot.shape)  # genes x conditions
df_pivot.head(8)

In [ ]:
cond.head(3)

In [ ]:
cond["cell_line_id"].nunique(), cond["drug"].nunique()

In [ ]:
cond["cell_line_id"].value_counts()     # expect 7 lines

In [ ]:
cond["drug"].value_counts()


In [ ]:
print("\n".join(np.unique(cond["drug"])))

In [ ]:
## 2D-values, stacked distribution
df_pivot.stack().describe()

In [ ]:
np.sum(np.sum(df_pivot<=1))

In [ ]:
np.sum(np.sum(df_pivot<=-1))

In [ ]:
float((df_pivot <= -1).mean().mean())

In [ ]:
(df_pivot >= 1).mean(axis=0)

In [ ]:
(df_pivot >= 1).mean(axis=1)

### Cluster

#### All three checks pass

- 0.4989 negative confirms stat is a genuinely signed statistic 
- the WTCS sign convention is sound. 
- 1986 of 2000 HVGs found in Tahoe is 99.3% coverage, better than I expected for a Parse 3' assay.

#### Score it:

In [ ]:
X.head(3)

### I'd question whether you want to run it at all.

axis_modality rejected bimodality on both axes in both cohorts independently — that's the replicated finding. Clustering a continuum will still return groups (PAC minimises fine on arbitrary cuts), and every previous attempt gave degenerate splits: 6/119, 6/111, 3/108.

If you're running it as a check on the new deconvolution, worth passing what you learned:

In [ ]:
cc = mc.consensus_cluster(X)
cc

In [ ]:
mc.cluster_summary(cc)                       # read `sizes` and `min_frac`, not just PAC

In [ ]:
mc.choose_k(cc, min_cluster_frac=0.10)       # rejects degenerate solutions

### PAC

Proportion of Ambiguous Clustering — a measure of how decisively the consensus clustering assigns samples, from Șenbabaoğlu et al. (2014). 

It's how choose_k picks k.

The consensus matrix C[i,j] is the fraction of resamples in which samples i and j landed in the same cluster, given both were drawn. 

Perfect structure gives values of 0 or 1 — pairs always together or always apart. Unstable structure gives values scattered in between.

PAC is just the fraction of pairs sitting in that ambiguous middle:



In [ ]:
consensus = cc[2]['consensus']
consensus.iloc[:5, :10]


In [ ]:
def _pac(consensus, lo=0.1, hi=0.9):
    v = consensus[np.triu_indices_from(consensus, k=1)]
    return float(((v > lo) & (v < hi)).mean())

_pac(consensus.values, lo=0.1, hi=0.9)

Low PAC = crisp, reproducible partition. choose_k takes the smallest k within pac_tol of the minimum, preferring parsimony when several k are comparably stable.

Why it misled you here. PAC rewards reproducibility, not biological meaning, and those come apart badly for unbalanced splits. Peeling six outliers off 119 samples is maximally reproducible — every resample isolates them identically — so PAC approaches zero. The metric is behaving exactly as designed while pointing at nothing interesting.

It's also mechanically biased toward small k, since fewer clusters means fewer boundaries to disagree about. That's why choose_k at k=2 deserves scepticism rather than confidence on its own.

So read the diagnostics together:

In [ ]:
dfa = pd.DataFrame({k: {"pac": v["pac"], "coph": v["cophenetic"],
                    "sil": v["silhouette"],
                    "sizes": v["labels"].value_counts().tolist()}
                    for k, v in cc.items()}).T

dfa

The sizes column is the one that would have caught this. 

A k with low PAC and balanced clusters is trustworthy; 
low PAC with a 6/119 split is an outlier detector. 

Cophenetic correlation and silhouette are worth glancing at too, though both share the same blind spot — none of them knows the difference between a real subtype and six weird samples.

In [ ]:
k = mc.choose_k(cc)
k

### k=2 is the expected answer for PDAC 

- Moffitt's classical vs basal-like is a two-group axis. 
- The earlier problem wasn't k=2, it was the 6-vs-119 split. 
- So the question now is what the two groups are.

Three checks, in order of how much they'd change your interpretation:

In [ ]:
labels = cc[2]["labels"]
print('n counts', labels.value_counts().to_dict())          # balanced now?
print("")
print(diag["pc_theta_pearson"])                 # PC1 vs theta_mal
print("")
print(pd.crosstab(labels, pd.Series(
    ['TCGA' if 'TCGA' in s else 'CPTAC' for s in X.index], index=X.index)))

A near-even split with |r| below ~0.3 on PC1 is what you want.

- If the crosstab shows the split tracking TCGA vs CPTAC, it's a batch axis — plausible given your strandedness history, 
- and it would mean the unstranded harmonisation didn't fully remove the cohort effect.

Then the test that actually names the clusters:

In [ ]:
basal = ["KRT81","KRT5","KRT6A","KRT17","S100A2","SPRR3","TP63","DHRS9","VGLL1"]
clas  = ["GATA6","TFF1","TFF2","TFF3","LGALS4","CLDN18","CEACAM6","AGR2", "ANXA10","REG4","CTSE","MUC13"]
Z = (X - X.mean()) / X.std()
sc = pd.DataFrame({
    "basal":     Z[[g for g in basal if g in X.columns]].mean(axis=1),
    "classical": Z[[g for g in clas  if g in X.columns]].mean(axis=1)})
print(sc.groupby(labels).mean().round(2))

If **one cluster is basal-high/classical-low** and **the other the reverse**, you've recovered Moffitt in the deconvolved malignant compartment

- which is a genuinely stronger result than the bulk clustering you started with, 
- because it's not confounded by stromal content. That was the whole point of the deconvolution detour.

If instead both clusters co-elevate the two programs, 
- you're seeing the same cellularity axis as before, and the purity decoupling didn't clear it.

Note how few of those markers likely survived your HVG filter — check [g for g in basal+clas if g in X.columns] first. If coverage is thin, score on the un-HVG-filtered logx instead, since marker scoring doesn't need the variance selection.

In [ ]:
labels = cc[k]["labels"]
sig    = mc.cluster_signatures(X, labels)
sig[1]

In [ ]:
"; ".join(sig[1]['stat'].index.to_list())

In [ ]:
"; ".join(sig[2]['stat'].index.to_list())

p = 4.8e-48 with n=6 is not credible. Welch t with ~5 df cannot produce that unless the within-group variance of the 6 is near zero. That's the fingerprint of prior domination: BayesPrism shrank all six toward the same reference profile, so they're nearly identical to each other. Tiny SE → exploding t → absurd p. Check directly:

In [ ]:
small = labels[labels==1].index
big   = labels[labels==2].index
pd.DataFrame({
    "sd_small": X.loc[small].std(axis=0),
    "sd_big":   X.loc[big].std(axis=0),
}).describe().round(3)

### Welch test

In [ ]:
labs1 = labels[labels==1].index
labs2 = labels[labels==2].index

X1 = X.loc[labs1]
X2 = X.loc[labs2]

X1.shape, X2.shape

In [ ]:
X1.iloc[:5, :10]

In [ ]:
X1.mean(axis=0)

In [ ]:
m1, m2 = X1.mean(axis=0), X2.mean(axis=0)      # Series, one value per gene
v1, v2 = X1.var(axis=0, ddof=1), X2.var(axis=0, ddof=1)
n1, n2 = len(X1), len(X2)

In [ ]:
from scipy import stats

se    = np.sqrt(v1/n1 + v2/n2)
tw    = (m1 - m2) / se
dfree = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
# Survival function (also defined as 1 - cdf, but sf is sometimes more accurate).
p = 2 * stats.t.sf(np.abs(tw), dfree)
p

In [ ]:
pd.DataFrame({
    "p": p,
}).describe()

In [ ]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')

fdr_cutoff = 0.05
lfc_cutoff = 1

dft_all = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.lfc.abs() >= lfc_cutoff)]
print(dft_all.shape)

dft_all.head(2)

In [ ]:
p = dft_all.loc[dft_all.cluster == 1, "p_nominal"]
p.describe()

In [ ]:
p.hist()

### Review clusters - detailed

In [ ]:
small_out = labels[labels == 1].index
X2, d2 = mc.prepare_malignant_matrix(
    keep_genes=mc.program1_panel, drop_pattern=r"^N-",
    keep_samples=[s for s in mc.ms.Z.index if s not in set(small_out)])
cc2 = mc.consensus_cluster(X2)
k2  = mc.choose_k(cc2)
print(k2) 
cc2[k2]["labels"].value_counts().to_dict()

In [ ]:
mc.cluster_summary(cc2)

If min_frac is tiny at every k, don't pick a k. Find the QC axis instead:

In [ ]:
d2["sample_total_Z"].sort_values().head(15)
mc.ms.theta_mal[X2.index].sort_values().head(15)
ties = X2.round(6).apply(lambda c: c.duplicated(keep=False)).mean(axis=1)
ties.sort_values(ascending=False).head(15)

### Critic

In your notebook, cell 72's output showed 'n': 6 and 'n': 119 - the n field cluster_signatures records for each cluster. 

So choose_k picked k=2, but the two groups were 6 samples and 119 samples, not two comparable halves.

In [ ]:
sig[1]['n'], sig[2]['n']

### Each signature

In [ ]:

clu=2
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
clu=1
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
sig[clu]['up']

### signature_table()

In [ ]:
len(labels), labels

In [ ]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')
dft['abs_lfc'] = dft['lfc'].abs()

fdr_cutoff = 0.05
lfc_cutoff = 1

dft = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.abs_lfc >= lfc_cutoff)]
dft = dft.sort_values('fdr_nominal', ascending=True)

dft.shape

In [ ]:
dft.fdr_nominal.hist()

In [ ]:
dft.head(6)

In [ ]:
genes_sel = sig[clu]['up']

df1 = dft[dft.gene.isin(genes_sel)]
print(f"Number of upregulated genes in cluster {clu}: {df1.shape[0]}")
df1

In [ ]:
genes_clu = np.unique(df1.gene)
print(len(genes_clu))


In [ ]:
genes_clu_in = [x for x in genes_clu if x in df_pivot.index]
df2 = df_pivot.loc[genes_clu_in]

print(df2.shape)
df2.T

### WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score)

WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score) are core metrics used in the LINCS and Connectivity Map (CMap) pipelines to compare query gene signatures against reference expression profiles. 

- WTCS measures signature similarity from −1 to 1
- NCS normalizes these scores within specific cell lines and perturbagen types.

In [ ]:
cond

In [ ]:
cond.index.is_unique 

In [ ]:
cond = cond[~cond.index.duplicated()].loc[df_pivot.columns]
print(cond.index.is_unique)
cond.head(3)

In [ ]:
R1 = mc.score_clusters_vs_tahoe(sig, df_pivot, cond)
R1

In [ ]:
len(R1.targets.unique()), R1.targets.unique()[:20]

In [ ]:
target_list = R1.targets.unique()
len(target_list), len(genes_clu)

In [ ]:
[x for x in target_list if x in genes_clu]

In [ ]:
R1a = R1[ (R1.cluster == clu) & (R1.targets.isin(genes_clu)) & (R1.wtcs.abs() > 0.1) ]
print(R1a.shape)
R1a

In [ ]:
R1a.targets.unique()

In [ ]:
R1a.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R1a.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
out = mc.run(ks=range(2, 9), drop_pattern=r"^N-", min_share=0.3)     # tune from diagnose_filters()
R   = mc.score_clusters_vs_tahoe(out["signatures"], df_pivot, cond)

In [ ]:
R.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
# mc.root_tahoe = create_dir(mc.root_mprog_cluster / "tahoe")
mc.root_tahoe

In [ ]:
d = mc.root_tahoe / "metadata" / "pseudobulk_differential_expression"
files = sorted(d.glob("*.parquet"))
len(files), sum(f.stat().st_size for f in files) / 1e9

In [ ]:
cl = pd.read_parquet(mc.root_tahoe / "metadata"/ "cell_line_metadata.parquet")
cl[cl.Organ=="Pancreas"][["Cell_ID_Cellosaur","cell_name"]]

In [ ]:
d = mc.diagnose_filters()

d["Z_looks_like_counts"], d["Z_median_of_medians"]


In [ ]:
d["by_min_counts"]

In [ ]:
d["by_min_share"]

In [ ]:
d["joint_grid"]